# BookLM: полный прямой проход от token_ids до logits

Большую конфигурацию BookLM нельзя честно выписать поэлементно, поэтому сначала соберём геометрически подобный стенд: словарь 8, ширина 4, две головы запросов, одна голова K/V, два блока и SwiGLU ширины 6. Имена осей и порядок операций останутся теми же, что у модели на 6 622 464 параметра.

In [ ]:
import hashlib
import json
import math
import platform

print('python:', platform.python_version())

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def matvec(matrix, vector):
    return [dot(row, vector) for row in matrix]

def add(left, right):
    return [a + b for a, b in zip(left, right)]

def stable_softmax(values):
    maximum = max(values)
    weights = [math.exp(value - maximum) for value in values]
    total = sum(weights)
    return [weight / total for weight in weights]

def weighted_sum(weights, vectors):
    return [sum(weight * vector[column] for weight, vector in zip(weights, vectors))
            for column in range(len(vectors[0]))]

def rmsnorm(vector, weight, epsilon=1e-6):
    inverse_rms = 1.0 / math.sqrt(sum(value * value for value in vector) / len(vector) + epsilon)
    return [value * inverse_rms * scale for value, scale in zip(vector, weight)]

def silu(value):
    return value / (1.0 + math.exp(-value))

def deterministic_matrix(rows, columns, seed):
    return [[((((row + 1) * (column + 3) + seed * 5) % 17) - 8) / 20
             for column in range(columns)] for row in range(rows)]

def shape_2d(values):
    return (len(values), len(values[0]) if values else 0)

## 1. Паспорт и параметры маленького стенда

W_Q выдаёт Hq·Dh=4 координаты, а W_K/W_V — только Hkv·Dh=2. Поэтому обе головы запросов будут читать одну голову ключей и значений: это GQA, а не уменьшенная запись обычного MHA.

In [ ]:
TOY = {
    'vocab_size': 8,
    'd_model': 4,
    'n_layers': 2,
    'n_heads': 2,
    'n_kv_heads': 1,
    'd_ff': 6,
}
assert TOY['d_model'] % TOY['n_heads'] == 0
HEAD_DIM = TOY['d_model'] // TOY['n_heads']
KV_WIDTH = TOY['n_kv_heads'] * HEAD_DIM
GROUP_SIZE = TOY['n_heads'] // TOY['n_kv_heads']
assert (HEAD_DIM, KV_WIDTH, GROUP_SIZE) == (2, 2, 2)

embeddings = deterministic_matrix(TOY['vocab_size'], TOY['d_model'], seed=1)

def make_block(seed):
    d, f = TOY['d_model'], TOY['d_ff']
    return {
        'norm_attn': [1.0] * d,
        'wq': deterministic_matrix(d, d, seed + 1),
        'wk': deterministic_matrix(KV_WIDTH, d, seed + 2),
        'wv': deterministic_matrix(KV_WIDTH, d, seed + 3),
        'wo': deterministic_matrix(d, d, seed + 4),
        'norm_mlp': [1.0] * d,
        'w_gate': deterministic_matrix(f, d, seed + 5),
        'w_up': deterministic_matrix(f, d, seed + 6),
        'w_down': deterministic_matrix(d, f, seed + 7),
    }

blocks = [make_block(10), make_block(30)]
final_norm_weight = [1.0] * TOY['d_model']

## 2. RoPE и GQA внутри одного блока

Пара координат головы поворачивается на угол, зависящий от позиции. При Hq=2 и Hkv=1 обе query-головы используют один и тот же набор K/V, но имеют разные проекции Q. Причинность задаётся диапазоном source=0…position.

In [ ]:
def rope(pair, position):
    assert len(pair) == 2
    angle = position * 0.2
    cosine, sine = math.cos(angle), math.sin(angle)
    x, y = pair
    return [x * cosine - y * sine, x * sine + y * cosine]

def gqa_attention(sequence, block, position_mode='correct'):
    normalized = [rmsnorm(state, block['norm_attn']) for state in sequence]
    queries = [matvec(block['wq'], state) for state in normalized]
    keys = [matvec(block['wk'], state) for state in normalized]
    values = [matvec(block['wv'], state) for state in normalized]

    rotated_keys = []
    for position, key in enumerate(keys):
        rope_position = position if position_mode == 'correct' else 0
        rotated_keys.append(rope(key, rope_position))

    projected_outputs = []
    for position, query_vector in enumerate(queries):
        heads = []
        for query_head in range(TOY['n_heads']):
            start = query_head * HEAD_DIM
            rope_position = position if position_mode == 'correct' else 0
            query = rope(query_vector[start:start + HEAD_DIM], rope_position)
            # query_head // GROUP_SIZE равно 0 для обеих голов: общий KV-владелец.
            kv_head = query_head // GROUP_SIZE
            assert kv_head == 0
            allowed_keys = rotated_keys[:position + 1]
            allowed_values = values[:position + 1]
            scores = [dot(query, key) / math.sqrt(HEAD_DIM) for key in allowed_keys]
            heads.extend(weighted_sum(stable_softmax(scores), allowed_values))
        projected_outputs.append(matvec(block['wo'], heads))
    return projected_outputs

def swiglu(sequence, block):
    outputs = []
    for state in sequence:
        normalized = rmsnorm(state, block['norm_mlp'])
        gate = matvec(block['w_gate'], normalized)
        up = matvec(block['w_up'], normalized)
        hidden = [silu(gate_value) * up_value for gate_value, up_value in zip(gate, up)]
        outputs.append(matvec(block['w_down'], hidden))
    return outputs

def transformer_block(sequence, block, position_mode='correct'):
    attention_update = gqa_attention(sequence, block, position_mode=position_mode)
    after_attention = [add(state, update) for state, update in zip(sequence, attention_update)]
    mlp_update = swiglu(after_attention, block)
    return [add(state, update) for state, update in zip(after_attention, mlp_update)]

## 3. Полный маршрут token_ids → embeddings → blocks → logits

Выходная матрица разделяет параметры с embedding: логит токена v — скалярное произведение финального состояния и строки embeddings[v]. Forward возвращает трассу форм, чтобы ни одно изменение осей не оставалось скрытым.

In [ ]:
def booklm_forward(token_ids, position_mode='correct'):
    assert token_ids and all(0 <= token_id < TOY['vocab_size'] for token_id in token_ids)
    states = [embeddings[token_id][:] for token_id in token_ids]
    trace = [('embeddings', shape_2d(states))]
    for layer_index, block in enumerate(blocks):
        states = transformer_block(states, block, position_mode=position_mode)
        trace.append((f'block_{layer_index}', shape_2d(states)))
    states = [rmsnorm(state, final_norm_weight) for state in states]
    logits = [[dot(state, token_embedding) for token_embedding in embeddings]
              for state in states]
    trace.append(('logits', shape_2d(logits)))
    return logits, trace

token_ids = [1, 4, 2, 7]
logits, trace = booklm_forward(token_ids)
assert trace == [
    ('embeddings', (4, 4)),
    ('block_0', (4, 4)),
    ('block_1', (4, 4)),
    ('logits', (4, 8)),
]
next_token_id = max(range(TOY['vocab_size']), key=lambda token_id: logits[-1][token_id])
print('trace:', trace)
print('last logits:', [round(value, 6) for value in logits[-1]])
print('greedy next token:', next_token_id)

## 4. Два независимых инварианта: причинность и позиция

Сначала изменяем последний token_id: прошлые логиты обязаны совпасть. Затем оставляем токены прежними, но ошибочно назначаем всем позицию ноль. Первая строка остаётся правильной, а последующие расходятся — формы этого не обнаруживают.

In [ ]:
mutated_logits, _ = booklm_forward([1, 4, 2, 3])
for reference_row, mutated_row in zip(logits[:-1], mutated_logits[:-1]):
    assert all(abs(a - b) < 1e-12 for a, b in zip(reference_row, mutated_row))

zero_position_logits, zero_trace = booklm_forward(token_ids, position_mode='zero')
assert zero_trace == trace
assert all(abs(a - b) < 1e-12 for a, b in zip(logits[0], zero_position_logits[0]))
position_error = max(abs(a - b)
                     for correct_row, wrong_row in zip(logits[1:], zero_position_logits[1:])
                     for a, b in zip(correct_row, wrong_row))
assert position_error > 1e-6
print('ошибка при position=0:', round(position_error, 8))

## 5. Тот же подсчёт для эталонной BookLM

Здесь мы не создаём миллионы чисел, а считаем строки матриц из паспорта. Выходная матрица не прибавляется второй раз, потому что веса связаны с embedding.

In [ ]:
BOOKLM = {
    'vocab_size': 8192,
    'd_model': 256,
    'n_layers': 6,
    'n_heads': 8,
    'n_kv_heads': 2,
    'd_ff': 768,
    'max_seq_len': 512,
    'tie_embeddings': True,
}
d = BOOKLM['d_model']
head_dim = d // BOOKLM['n_heads']
kv_width = BOOKLM['n_kv_heads'] * head_dim
embedding_parameters = BOOKLM['vocab_size'] * d
attention_parameters = d * d + 2 * d * kv_width + d * d
swiglu_parameters = 3 * d * BOOKLM['d_ff']
norm_parameters = 2 * d
total_parameters = (embedding_parameters
                    + BOOKLM['n_layers'] * (attention_parameters + swiglu_parameters + norm_parameters)
                    + d)
assert total_parameters == 6_622_464

manifest_bytes = json.dumps(BOOKLM, sort_keys=True, separators=(',', ':')).encode('utf-8')
print('parameters:', total_parameters)
print('config sha256:', hashlib.sha256(manifest_bytes).hexdigest())

## 6. Код главы 5 из `labs/ch05`

Полные версии трёх длинных листингов главы 5 лежат в отдельных файлах `labs/ch05/*.py`. В книге остались сокращённые версии, а здесь каждый файл выполняется целиком: при загрузке он проходит все свои `assert` и печатает строку «все проверки пройдены».

Файлы загружаются через `importlib` по явному пути. Каждый получает своё пространство имён, поэтому его функции не перекрывают функции стенда выше. Путь ищется от текущего каталога вверх, так что ноутбук работает и из `notebooks/`, и из корня репозитория.

In [ ]:
import importlib.util
from pathlib import Path


def find_labs_ch05():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / 'labs' / 'ch05'
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError('labs/ch05 не найден: запустите ноутбук внутри репозитория книги')


LABS_CH05 = find_labs_ch05()


def load_lab(name):
    lab_path = LABS_CH05 / f'{name}.py'
    spec = importlib.util.spec_from_file_location(f'ch05_{name}', lab_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)   # здесь выполняются все проверки файла
    return module


print('labs:', LABS_CH05)

### 6.1. Паспорт модели из `config.json`

Механизм раздела «Читаем config.json как паспорт BookLM» и подраздела «Общее и активное число параметров»: по полям `config.json` считаем число параметров и размер KV-кэша, не загружая весов. Файл `labs/ch05/architecture_passport.py` проверяет BookLM, Mistral 7B, Mixtral 8x7B, вариант BookLM со смесью экспертов и случай, когда в файле нет поля `tie_word_embeddings`.

Ниже сверяем паспорт с ручным подсчётом из раздела 5: обе дороги должны дать одно число.

In [ ]:
passport = load_lab('architecture_passport')

assert passport.parameter_count(passport.booklm) == total_parameters
assert passport.booklm['H_kv'] == BOOKLM['n_kv_heads']
total_moe = passport.moe_parameter_count(passport.booklm_moe)
active_moe = passport.moe_parameter_count(passport.booklm_moe, active=True)
assert active_moe < total_moe
print('BookLM:', passport.parameter_count(passport.booklm))
print('BookLM-MoE, всего / активно:', total_moe, '/', active_moe)
print('KV-кэш BookLM на 512 токенов, КБ:', passport.kv_cache_bytes(passport.booklm, 512) // 1024)

### 6.2. Слой смеси экспертов

Механизм раздела «Смесь экспертов: все веса хранятся, часть работает»: маршрутизатор выбирает k экспертов из E, веса считаются softmax только по выбранным, а групповой проход вызывает каждого эксперта один раз на все его токены. Файл `labs/ch05/moe_layer.py` берёт в эксперты настоящие SwiGLU-ветви и сверяет слой с наивным циклом токен за токеном.

In [ ]:
moe_lab = load_lab('moe_layer')

expert_parameters = 3 * 256 * 768
assert moe_lab.n_params == 4 * expert_parameters + 256 * 4
print('параметров в слое:', moe_lab.n_params)
print('из них работает на токен:', 2 * expert_parameters + 256 * 4)

### 6.3. Ёмкость эксперта и балансирующая ошибка

Механизм подраздела «Ёмкость и балансировка»: у эксперта ограниченное число мест в пакете, лишние назначения отбрасываются, а балансирующая ошибка наказывает маршрутизатор за перекос. Файл `labs/ch05/moe_capacity_balance.py` добавляет к листингу книги коэффициент ёмкости 1,25 и проверку того, что градиент ошибки идёт только через средние вероятности P.

**Диагностика.** Поменяйте порядок строк в `chosen` и снова вызовите `dispatch`: какие назначения теперь отбрасываются и почему число отброшенных не изменилось?

In [ ]:
balance = load_lab('moe_capacity_balance')

for factor in (1.0, 1.25, 2.0):
    cap = balance.capacity(8, 2, 4, factor=factor)
    kept, load = balance.dispatch(balance.chosen, 4, cap)
    print(f'factor={factor}: мест на эксперта {cap}, загрузка {load}, отброшено {16 - len(kept)}')
assert 16 - len(balance.dispatch(balance.chosen, 4, balance.capacity(8, 2, 4, factor=2.0))[0]) == 0

## Переход к следующим главам

Состояния blocks станут обучаемыми в главе 7; к ним добавятся LoRA в главе 8, модальные порты в главе 6 и KV-кэш в главе 14. Все эти изменения должны сохранять исходный текстовый forward в объявленном режиме.

**Задание.** Добавьте пошаговый KV-кэш к функции gqa_attention и сравните последний логит каждого шага с полным forward.